In [1]:
import dspy
from dspy import ChainOfThought, InputField, Module, OutputField, Signature

In [2]:
lm = dspy.LM(
    "ollama_chat/gemma3:12b",
    api_base="http://localhost:11434",
    api_key="",
    max_tokens=40960,
    temperature=0.0,
    cache=False,
)
dspy.configure(lm=lm)

## Use DSPy built-in Module to Build a Sentiment Classifier

In [3]:
class SentimentClassifier(dspy.Signature):
    """Classify the sentiment of a text."""

    text: str = dspy.InputField(desc="input text to classify sentiment")
    sentiment: int = dspy.OutputField(
        desc="sentiment, the higher the more positive",
        ge=0,
        le=10,
    )


SentimentClassifier

SentimentClassifier(text -> sentiment
    instructions='Classify the sentiment of a text.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'input text to classify sentiment', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    sentiment = Field(annotation=int required=True json_schema_extra={'desc': 'sentiment, the higher the more positive', '__dspy_field_type': 'output', 'constraints': 'greater than or equal to: 0, less than or equal to: 10', 'prefix': 'Sentiment:'} metadata=[Ge(ge=0), Le(le=10)])
)

In [4]:
str_signature = dspy.make_signature("text -> sentiment")
str_signature

StringSignature(text -> sentiment
    instructions='Given the fields `text`, produce the fields `sentiment`.'
    text = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Text:', 'desc': '${text}'})
    sentiment = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Sentiment:', 'desc': '${sentiment}'})
)

### Create a Module to Interact with the LM

In [5]:
predict = dspy.Predict(SentimentClassifier)
predict

Predict(SentimentClassifier(text -> sentiment
    instructions='Classify the sentiment of a text.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'input text to classify sentiment', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    sentiment = Field(annotation=int required=True json_schema_extra={'desc': 'sentiment, the higher the more positive', '__dspy_field_type': 'output', 'constraints': 'greater than or equal to: 0, less than or equal to: 10', 'prefix': 'Sentiment:'} metadata=[Ge(ge=0), Le(le=10)])
))

In [6]:
output = predict(text="GODLIKE!")
output

Prediction(
    sentiment=10
)

### Wait, Where is My Prompt? 

In [7]:
dspy.inspect_history(n=1)





[2025-07-14T21:33:38.736635]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
GODLIKE!

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
10
[[ ## completed ## ]]







In [8]:
cot = dspy.ChainOfThought(SentimentClassifier)

output = cot(text="I am feeling somewhat happy!")
output

Prediction(
    reasoning='The text explicitly states "I am feeling happy," which is a positive emotion. The use of "somewhat" indicates a moderate level of happiness rather than extreme joy.',
    sentiment=6
)

In [9]:
dspy.inspect_history(n=1)





[2025-07-14T21:33:43.431598]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling somewhat happy!

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## sentiment ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]


In [10]:
dspy.configure(adapter=dspy.JSONAdapter())

In [11]:
output = cot(text="I am feeling somewhat happy!")
output

2025/07/14 21:33:43 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Prediction(
    reasoning="The text explicitly states 'happy,' indicating a positive sentiment. The use of 'somewhat' suggests a moderate level of happiness.",
    sentiment=6
)

In [12]:
dspy.inspect_history(n=1)





[2025-07-14T21:33:47.194048]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "sentiment": "{sentiment}        # note: the value you produce must be a single int value"
}
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling somewhat happy!

Respond with a JSON object in the following order of fields: `reasoning`, then `sentiment` (must be formatted as a valid Python int).


Response:

{"reasoning": "The text explicitly states 'h

## Build a Program with Custom Module

In [13]:
class QuestionGenerator(Signature):
    """
    Generate a yes or no question in order to guess the celebrity name on user's mind.
    You can ask in general or directly guess the name if you think the signal is enough.
    You should never ask or rephrase the question that was already asked.

    Go from broad questions to narrow ones, but at the same time ask as broad question as possible in current context
        - example: is male? (y) -> is connected to art? (y) -> is connected to audio art? (y) -> is singer? (y) -> ...)
    Don't try to guess too early
        - example: is male? (y) -> is musician? (n) -> is actor? (n) -> is singer? (n) -> is artist? (n) -> ...)

    Stay in the context of previous correct guesses
        - example: works in entertainment? (y) -> is connected to music? (y)

    Stay away from the context of previous incorrect guesses
        - example: works in entertainment? (n) -> is connected to music? (obviously no, since the music is a part of entertainment)
        - example: is male? (n) -> ... -> Is Bill Gates? (obviously no, since he is male and we said earlier that the selebrity is not male)

    """

    past_questions: list[str] = InputField(desc="past questions asked")
    past_answers: list[bool] = InputField(desc="past answers")

    new_question: str = OutputField(desc="new question that can help narrow down the celebrity name")
    guess_made: bool = OutputField(
        desc="If new_question is the celebrity name quess, set to True. If it is still a general question set to False"
    )


class Reflection(Signature):
    """Provide reflection on the quessing process"""

    correct_celebrity_name: str = InputField(desc="the celebrity name in user's mind")
    final_guessor_question: str = InputField(desc="the final guess or question LM made")
    past_questions: list[str] = InputField(desc="past questions asked")
    past_answers: list[bool] = InputField(desc="past answers")

    reflection: str = OutputField(
        desc="reflection on the guessing process, including what was done well and what can be improved"
    )


def ask(prompt, valid_responses=("y", "n")):
    while True:
        response = input(f"{prompt} ({'/'.join(valid_responses)}): ").strip().lower()
        if response in valid_responses:
            return response.lower()
        print(f"Please enter one of: {', '.join(valid_responses)}")


class CelebrityGuess(Module):
    def __init__(self, max_tries=10):
        super().__init__()

        self.max_tries = max_tries
        self.question_generator = ChainOfThought(QuestionGenerator)
        self.reflection = ChainOfThought(Reflection)

    def forward(self):
        celebrity_name = input("Please think of a celebrity name, once you are ready - type the name and press enter")
        past_questions = []
        past_answers = []

        correct_guess = False

        for i in range(self.max_tries):
            question = self.question_generator(
                past_questions=past_questions,
                past_answers=past_answers,
            )
            answer = ask(f"{question.new_question}") == "y"
            print(f"question={question.new_question}, {answer=}")
            past_questions.append(question.new_question)
            past_answers.append(answer)

            if question.guess_made and answer:
                correct_guess = True
                break

        if correct_guess:
            print("Yay! I got it right!")
        else:
            print("Oops, I couldn't guess it right.")

        reflection = self.reflection(
            correct_celebrity_name=celebrity_name,
            final_guessor_question=question.new_question,
            past_questions=past_questions,
            past_answers=past_answers,
        )

        print(reflection.reflection)

In [14]:
celebrity_guess = CelebrityGuess()

In [15]:
celebrity_guess()

2025/07/14 21:34:04 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/07/14 21:34:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


question=Are they primarily known for their work in entertainment?, answer=False


2025/07/14 21:34:35 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


question=Are they known for their work in politics?, answer=False


2025/07/14 21:34:46 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


...

question=Are they primarily known for their work in software development?, answer=True


2025/07/14 21:35:46 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


question=Are they primarily known for developing web-based applications?, answer=False


2025/07/14 21:35:57 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


question=Are they known for their work on operating systems?, answer=False
Oops, I couldn't guess it right.
The guessing process was effective in eliminating irrelevant categories. Starting with broad categories (entertainment, politics, sports) and then moving to more specific areas (science/tech, computer science, software development) proved to be a good strategy. The inclusion of a 'still alive?' question was helpful in narrowing the field. A potential improvement would be to incorporate questions about the *type* of software development (e.g., low-level, high-level, scripting languages) earlier in the process to further refine the possibilities. Also, asking about programming languages specifically might have led to a quicker identification.


In [16]:
celebrity_guess.save("../temp/celebrity.json", save_program=False)